In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 73 — Production Search Assistant (Haystack)

## Goal
Build a **scalable search + answer system** using
Haystack's **pipeline** and **component** architecture.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Haystack Pipeline** | DAG of connected components |
| **Document Store** | In-memory store for documents |
| **BM25 Retriever** | Keyword-based retrieval |
| **Embedding Retriever** | Semantic similarity search |
| **PromptBuilder** | Template-based prompt construction |
| **OllamaChatGenerator** | Local LLM generation via Ollama |

## Stack
- `haystack-ai` 2.27+
- `ollama-haystack` (OllamaChatGenerator)
- Ollama `qwen3.5:9b`

In [ ]:
import os, warnings
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from haystack import Pipeline, Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
from haystack.components.builders import PromptBuilder
from haystack_integrations.components.generators.ollama import OllamaChatGenerator
from haystack.dataclasses import ChatMessage

print("All imports OK")

## Step 1 — Create Document Store

Haystack's `InMemoryDocumentStore` is perfect for
prototyping. In production, swap in Elasticsearch,
Weaviate, Qdrant, etc.

In [ ]:
# Create document store and populate with tech documents
doc_store = InMemoryDocumentStore()

documents = [
    Document(content="""FastAPI is a modern Python web framework for building APIs.
It is built on top of Starlette (ASGI) and Pydantic (data validation).
Key features: automatic OpenAPI docs, async support, type hints for validation,
dependency injection. Performance is comparable to Node.js and Go.
FastAPI uses Python type hints to automatically validate request/response data.""",
             meta={"topic": "web_framework", "name": "FastAPI"}),
    
    Document(content="""Django is a high-level Python web framework that encourages
rapid development. It follows the MTV (Model-Template-View) pattern.
Built-in features: admin panel, ORM, authentication, form handling, middleware.
Django REST Framework extends it for building APIs. Used by Instagram, Pinterest.
Philosophy: 'batteries included' — everything needed comes out of the box.""",
             meta={"topic": "web_framework", "name": "Django"}),
    
    Document(content="""PostgreSQL is an advanced open-source relational database.
Features: ACID compliance, JSON/JSONB support, full-text search, window functions,
CTEs, materialized views, table partitioning, logical replication.
Extensions: PostGIS (geospatial), TimescaleDB (time-series), pgvector (embeddings).
PostgreSQL supports both SQL and NoSQL workloads effectively.""",
             meta={"topic": "database", "name": "PostgreSQL"}),
    
    Document(content="""Redis is an in-memory data structure store used as database,
cache, and message broker. Data structures: strings, hashes, lists, sets, sorted
sets, streams, bitmaps. Use cases: session caching, real-time leaderboards,
pub/sub messaging, rate limiting. Redis Cluster provides horizontal scaling.
Persistence options: RDB snapshots and AOF (Append Only File).""",
             meta={"topic": "database", "name": "Redis"}),
    
    Document(content="""Kubernetes (K8s) is a container orchestration platform.
Core concepts: Pods, Deployments, Services, Ingress, ConfigMaps, Secrets.
Features: auto-scaling, self-healing, rolling updates, service discovery.
Architecture: control plane (API server, scheduler, etcd) + worker nodes (kubelet).
Ecosystem: Helm (package manager), Istio (service mesh), ArgoCD (GitOps).""",
             meta={"topic": "devops", "name": "Kubernetes"}),
    
    Document(content="""Docker is a platform for containerizing applications.
Containers package code + dependencies into portable units. Key components:
Dockerfile (build instructions), Images (read-only templates), Containers
(running instances), Volumes (persistent storage), Networks.
Docker Compose manages multi-container applications with a YAML file.
Best practices: multi-stage builds, non-root users, .dockerignore.""",
             meta={"topic": "devops", "name": "Docker"}),
]

doc_store.write_documents(documents)
print(f"Document store: {doc_store.count_documents()} documents")
for doc in documents:
    print(f"  - [{doc.meta['topic']}] {doc.meta['name']}")

## Step 2 — Build a Basic RAG Pipeline

A Haystack pipeline is a **directed acyclic graph** (DAG)
where components are connected by named inputs/outputs.

In [ ]:
# Define the prompt template
prompt_template = """
You are a helpful tech assistant. Answer the question based on the provided context.
If the context doesn't contain enough information, say so.

Context:
{% for doc in documents %}
  - [{{ doc.meta.name }}]: {{ doc.content }}
{% endfor %}

Question: {{ question }}

Answer concisely and cite which technology you're referring to.
"""

# Create pipeline components
retriever = InMemoryBM25Retriever(document_store=doc_store, top_k=3)
prompt_builder = PromptBuilder(template=prompt_template)
generator = OllamaChatGenerator(
    model="qwen3.5:9b",
    url="http://localhost:11434",
)

print("Components created:")
print(f"  Retriever: InMemoryBM25Retriever (top_k=3)")
print(f"  Prompt: PromptBuilder")
print(f"  Generator: OllamaChatGenerator (qwen3.5:9b)")

In [ ]:
# Build and connect the pipeline
rag_pipeline = Pipeline()

# Add components
rag_pipeline.add_component("retriever", retriever)
rag_pipeline.add_component("prompt_builder", prompt_builder)
rag_pipeline.add_component("llm", generator)

# Connect: retriever.documents → prompt_builder.documents
rag_pipeline.connect("retriever.documents", "prompt_builder.documents")
# Connect: prompt_builder output → llm (as user message)
# OllamaChatGenerator expects 'messages' input

print("Pipeline built!")
print("  retriever → prompt_builder → llm")

In [ ]:
def search_and_answer(question: str):
    """Run the RAG pipeline: retrieve → prompt → generate."""
    print(f"\n{'='*70}")
    print(f"Q: {question}")
    print(f"{'='*70}")
    
    # Step 1: Retrieve
    retrieval_result = retriever.run(query=question)
    docs = retrieval_result["documents"]
    print(f"\nRetrieved {len(docs)} documents:")
    for doc in docs:
        print(f"  - {doc.meta['name']} (score: {doc.score:.3f})")
    
    # Step 2: Build prompt
    prompt_result = prompt_builder.run(documents=docs, question=question)
    prompt_text = prompt_result["prompt"]
    
    # Step 3: Generate answer
    messages = [ChatMessage.from_user(prompt_text)]
    gen_result = generator.run(messages=messages)
    answer = gen_result["replies"][0].text
    
    print(f"\nAnswer:\n{answer[:600]}")
    return answer

print("search_and_answer() ready")

In [ ]:
# Test 1: Framework question
search_and_answer("What is FastAPI and how does it handle validation?")

In [ ]:
# Test 2: Cross-topic question
search_and_answer("How do Docker and Kubernetes work together?")

In [ ]:
# Test 3: Database comparison
search_and_answer("When should I use PostgreSQL vs Redis?")

## Step 3 — Adding a Keyword Filter

Let's add metadata filtering to narrow results
by topic before retrieval.

In [ ]:
def filtered_search(question: str, topic: str = None):
    """Search with optional topic filter."""
    print(f"\n{'='*70}")
    print(f"Q: {question}" + (f" [filter: {topic}]" if topic else ""))
    print(f"{'='*70}")
    
    if topic:
        filters = {"field": "meta.topic", "operator": "==", "value": topic}
        result = retriever.run(query=question, filters={"conditions": [filters], "operator": "AND"})
    else:
        result = retriever.run(query=question)
    
    docs = result["documents"]
    print(f"Retrieved {len(docs)} docs: {[d.meta['name'] for d in docs]}")
    
    if not docs:
        print("No documents found for this filter.")
        return
    
    prompt_result = prompt_builder.run(documents=docs, question=question)
    messages = [ChatMessage.from_user(prompt_result["prompt"])]
    gen_result = generator.run(messages=messages)
    print(f"\nAnswer:\n{gen_result['replies'][0].text[:500]}")

# Filtered: only database docs
filtered_search("What data structures are supported?", topic="database")

In [ ]:
# Filtered: only devops docs
filtered_search("How do I deploy my application?", topic="devops")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Pipeline** | DAG of components connected by input/output names |
| **Document** | `Document(content=..., meta={...})` |
| **DocumentStore** | `InMemoryDocumentStore` (swap for Elasticsearch/Qdrant) |
| **BM25 Retriever** | Keyword-based retrieval with TF-IDF scoring |
| **PromptBuilder** | Jinja2 template for prompt construction |
| **OllamaChatGenerator** | Local LLM via Ollama for generation |
| **Metadata filtering** | `filters=` parameter on retriever |

## Haystack Pipeline Architecture
```
User Query
    │
    ├─ Retriever          (BM25 / Embedding / Hybrid)
    │    ↓ documents
    ├─ PromptBuilder      (Jinja2 template)
    │    ↓ prompt
    ├─ Generator          (Ollama / OpenAI / HuggingFace)
    │    ↓ replies
    └─ Answer
```

## Production Extensions
- Swap `InMemoryDocumentStore` for `ElasticsearchDocumentStore` or `QdrantDocumentStore`
- Add `DocumentCleaner` and `DocumentSplitter` for preprocessing
- Add `SentenceTransformersDocumentEmbedder` for semantic search
- Use `Pipeline.draw()` to visualize your pipeline as a graph